# E-Commerce Analytics Project
## 01 — Raw Data Generation

### Objective
Create a realistic multi-table e-commerce dataset that will later be used for SQL querying, Python data cleaning and analysis, and an executive BI dashboard.

### Raw Tables
- Customers
- Products
- Orders
- Order Items
- Returns
- Marketing

The generated data will intentionally contain selected data-quality issues so that the cleaning and validation stages reflect a realistic data analyst workflow.

In [1]:
%pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 536.4 kB/s eta 0:00:001m1.5 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd 
import numpy as np
import random 
import sqlite3

from faker import Faker 
from pathlib import Path 

In [3]:
# Project directories

PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATABASE_DIR = PROJECT_ROOT / "database"

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DIR)
print("Processed data folder:", PROCESSED_DIR)
print("Database folder:", DATABASE_DIR)

Project root: /Users/gonzalojenkinsmadrid/E-Commerce-Analytics
Raw data folder: /Users/gonzalojenkinsmadrid/E-Commerce-Analytics/data/raw
Processed data folder: /Users/gonzalojenkinsmadrid/E-Commerce-Analytics/data/processed
Database folder: /Users/gonzalojenkinsmadrid/E-Commerce-Analytics/database


In [4]:
# Reproducibility

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

fake = Faker("en_US")
Faker.seed(SEED)


# Dataset period

START_DATE = pd.Timestamp("2023-01-01")
END_DATE = pd.Timestamp("2025-12-31")


# Dataset size

N_CUSTOMERS = 25_000
N_PRODUCTS = 150
N_ORDERS = 75_000


print(f"Analysis period: {START_DATE.date()} to {END_DATE.date()}")
print(f"Customers: {N_CUSTOMERS:,}")
print(f"Products: {N_PRODUCTS:,}")
print(f"Orders: {N_ORDERS:,}")

Analysis period: 2023-01-01 to 2025-12-31
Customers: 25,000
Products: 150
Orders: 75,000


## 1. Customer Data

The customer table represents individual customers registered with the e-commerce company.

Each customer will have:
- Unique customer ID
- Signup date
- City
- State
- Geographic region
- Age group
- Acquisition channel

In [5]:
states_regions = {
    "CA": "West",
    "WA": "West",
    "OR": "West",
    "NV": "West",
    "AZ": "West",
    "CO": "West",

    "TX": "South",
    "FL": "South",
    "GA": "South",
    "NC": "South",
    "VA": "South",

    "NY": "Northeast",
    "NJ": "Northeast",
    "MA": "Northeast",
    "PA": "Northeast",

    "IL": "Midwest",
    "OH": "Midwest",
    "MI": "Midwest",
    "MN": "Midwest",
    "WI": "Midwest"
}


acquisition_channels = [
    "Organic Search",
    "Paid Search",
    "Social Media",
    "Email",
    "Affiliate",
    "Direct"
]

In [6]:
customers = []

for i in range(1, N_CUSTOMERS + 1):

    state = random.choice(list(states_regions.keys()))

    signup_date = fake.date_between_dates(
        date_start=START_DATE,
        date_end=END_DATE
    )

    age = np.random.randint(18, 70)

    if age <= 24:
        age_group = "18-24"
    elif age <= 34:
        age_group = "25-34"
    elif age <= 44:
        age_group = "35-44"
    elif age <= 54:
        age_group = "45-54"
    else:
        age_group = "55+"

    customers.append({
        "customer_id": f"CUST-{i:05d}",
        "signup_date": signup_date,
        "city": fake.city(),
        "state": state,
        "region": states_regions[state],
        "age_group": age_group,
        "acquisition_channel": random.choice(acquisition_channels)
    })


customers_df = pd.DataFrame(customers)

print(f"Customers generated: {len(customers_df):,}")

Customers generated: 25,000


In [7]:
customers_df.head()

,customer_id,signup_date,city,state,region,age_group,acquisition_channel
0,CUST-00001,2024-12-01,Lake Joshuabury,NV,West,55+,Organic Search
1,CUST-00002,2023-06-02,Lake Joyside,GA,South,55+,Paid Search
2,CUST-00003,2023-04-06,Johnsonland,FL,South,45-54,Paid Search
3,CUST-00004,2023-08-28,New Jamesside,NV,West,25-34,Direct
4,CUST-00005,2024-04-04,Lawrencetown,MI,Midwest,55+,Organic Search


In [8]:
customers_df.shape

(25000, 7)

In [9]:
customers_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   customer_id          25000 non-null  object
 1   signup_date          25000 non-null  object
 2   city                 25000 non-null  object
 3   state                25000 non-null  object
 4   region               25000 non-null  object
 5   age_group            25000 non-null  object
 6   acquisition_channel  25000 non-null  object
dtypes: object(7)
memory usage: 1.3+ MB


## 2. Product Data

The product table represents the e-commerce company's product catalog.

Each product contains:
- Unique product ID
- Product name
- Category
- Subcategory
- Unit cost
- List price

Keeping product cost separate from selling price will allow us to analyze revenue, gross profit, margins, discounts, and product-level profitability.

In [10]:
catalog = {
    "Electronics": [
        "Headphones",
        "Smart Watch",
        "Bluetooth Speaker",
        "Keyboard",
        "Webcam"
    ],

    "Fashion": [
        "T-Shirt",
        "Hoodie",
        "Sneakers",
        "Jacket",
        "Backpack"
    ],

    "Home & Kitchen": [
        "Coffee Maker",
        "Cookware Set",
        "Blender",
        "Desk Lamp",
        "Storage Set"
    ],

    "Beauty & Health": [
        "Skincare Kit",
        "Hair Dryer",
        "Electric Toothbrush",
        "Face Serum",
        "Massage Gun"
    ],

    "Sports & Outdoors": [
        "Yoga Mat",
        "Water Bottle",
        "Running Shoes",
        "Resistance Bands",
        "Camping Light"
    ]
}

In [11]:
products = []

for i in range(1, N_PRODUCTS + 1):

    category = random.choice(list(catalog.keys()))
    subcategory = random.choice(catalog[category])

    # Generate the company's cost for the product
    unit_cost = round(
        np.random.uniform(8, 180),
        2
    )

    # Apply a realistic markup
    markup = np.random.uniform(
        1.3,
        2.4
    )

    list_price = round(
        unit_cost * markup,
        2
    )

    products.append({
        "product_id": f"PROD-{i:04d}",
        "product_name": f"{subcategory} {i}",
        "category": category,
        "subcategory": subcategory,
        "unit_cost": unit_cost,
        "list_price": list_price
    })


products_df = pd.DataFrame(products)

print(f"Products generated: {len(products_df):,}")

Products generated: 150


In [12]:
products_df.head(10)

,product_id,product_name,category,subcategory,unit_cost,list_price
0,PROD-0001,Massage Gun 1,Beauty & Health,Massage Gun,34.30,76.46
1,PROD-0002,Desk Lamp 2,Home & Kitchen,Desk Lamp,41.67,96.90
2,PROD-0003,Desk Lamp 3,Home & Kitchen,Desk Lamp,51.10,107.68
3,PROD-0004,Blender 4,Home & Kitchen,Blender,56.96,93.96
4,PROD-0005,Coffee Maker 5,Home & Kitchen,Coffee Maker,61.32,130.16
5,PROD-0006,Hair Dryer 6,Beauty & Health,Hair Dryer,103.77,235.69
6,PROD-0007,Water Bottle 7,Sports & Outdoors,Water Bottle,60.87,125.46
7,PROD-0008,Headphones 8,Electronics,Headphones,137.51,201.68
8,PROD-0009,Webcam 9,Electronics,Webcam,76.90,140.81
9,PROD-0010,Coffee Maker 10,Home & Kitchen,Coffee Maker,69.18,114.45


In [13]:
products_df.shape

(150, 6)

In [14]:
products_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    150 non-null    object 
 1   product_name  150 non-null    object 
 2   category      150 non-null    object 
 3   subcategory   150 non-null    object 
 4   unit_cost     150 non-null    float64
 5   list_price    150 non-null    float64
dtypes: float64(2), object(4)
memory usage: 7.2+ KB


In [15]:
products_df.groupby("category").agg(
    products=("product_id", "count"),
    avg_cost=("unit_cost", "mean"),
    avg_price=("list_price", "mean")
).round(2)

,products,avg_cost,avg_price
category,,,
Beauty & Health,34,102.90,197.08
Electronics,27,95.65,190.02
Fashion,24,69.91,127.48
Home & Kitchen,37,88.79,170.11
Sports & Outdoors,28,90.09,168.94


## 3. Order Data

The orders table represents individual purchases placed by customers.

Each order contains:
- Unique order ID
- Customer ID
- Order date
- Device used
- Sales channel
- Shipping cost
- Order status

Order dates will always occur on or after the customer's signup date.

The generated order volume will also include realistic seasonality and gradual business growth over time.

In [16]:
# Create all possible order dates

order_dates = pd.date_range(
    START_DATE,
    END_DATE,
    freq="D"
)

date_weights = []

for date in order_dates:

    # Gradual company growth over time
    days_since_start = (date - START_DATE).days
    growth_factor = 1 + (days_since_start / 1095) * 0.60

    # Seasonal e-commerce behavior
    if date.month in [11, 12]:
        seasonal_factor = 1.45
    elif date.month in [6, 7]:
        seasonal_factor = 1.10
    elif date.month in [1, 2]:
        seasonal_factor = 0.90
    else:
        seasonal_factor = 1.00

    # Slight weekend boost
    weekend_factor = 1.08 if date.weekday() >= 5 else 1.00

    weight = (
        growth_factor
        * seasonal_factor
        * weekend_factor
    )

    date_weights.append(weight)


date_weights = np.array(date_weights)

# Convert to probabilities
date_weights = date_weights / date_weights.sum()

In [25]:
# Customer signup lookup

customer_signup_lookup = (
    customers_df
    .assign(
        signup_date=pd.to_datetime(
            customers_df["signup_date"]
        )
    )
    .set_index("customer_id")["signup_date"]
)

orders = []

for i in range(1, N_ORDERS + 1):

    # 1. Choose the order date first
    order_date = pd.Timestamp(
        np.random.choice(
            order_dates,
            p=date_weights
        )
    )

    # 2. Identify customers who already existed on that date
    eligible_customers = (
        customer_signup_lookup[
            customer_signup_lookup <= order_date
        ]
        .index
        .tolist()
    )

    # If no customers existed yet, skip and retry
    if len(eligible_customers) == 0:
        continue

    # 3. Choose an eligible customer
    customer_id = random.choice(
        eligible_customers
    )

    device = random.choices(
        ["Mobile", "Desktop", "Tablet"],
        weights=[58, 36, 6]
    )[0]

    sales_channel = random.choices(
        ["Website", "Mobile App"],
        weights=[68, 32]
    )[0]

    shipping_cost = random.choices(
        [0.00, 4.99, 7.99, 12.99],
        weights=[35, 30, 25, 10]
    )[0]

    order_status = random.choices(
        ["Completed", "Cancelled"],
        weights=[97, 3]
    )[0]

    orders.append({
        "order_id": f"ORD-{i:06d}",
        "customer_id": customer_id,
        "order_date": order_date.date(),
        "device": device,
        "sales_channel": sales_channel,
        "shipping_cost": shipping_cost,
        "order_status": order_status
    })


orders_df = pd.DataFrame(orders)

print(f"Orders generated: {len(orders_df):,}")

Orders generated: 75,000


In [26]:
orders_df.head(10)

,order_id,customer_id,order_date,device,sales_channel,shipping_cost,order_status
0,ORD-000001,CUST-20202,2025-03-05,Desktop,Website,0.00,Completed
1,ORD-000002,CUST-18773,2024-12-05,Desktop,Website,0.00,Completed
2,ORD-000003,CUST-16907,2025-04-21,Mobile,Mobile App,7.99,Completed
3,ORD-000004,CUST-12557,2024-05-10,Mobile,Website,4.99,Completed
4,ORD-000005,CUST-01431,2025-06-01,Tablet,Website,12.99,Completed
5,ORD-000006,CUST-03625,2023-01-27,Mobile,Website,4.99,Completed
6,ORD-000007,CUST-01835,2023-09-27,Desktop,Website,7.99,Completed
7,ORD-000008,CUST-15251,2025-12-14,Mobile,Website,12.99,Completed
8,ORD-000009,CUST-19362,2023-05-15,Desktop,Website,4.99,Completed
9,ORD-000010,CUST-21480,2025-02-25,Mobile,Mobile App,0.00,Completed


In [27]:
orders_df.shape

(75000, 7)

In [28]:
orders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       75000 non-null  object 
 1   customer_id    75000 non-null  object 
 2   order_date     75000 non-null  object 
 3   device         75000 non-null  object 
 4   sales_channel  75000 non-null  object 
 5   shipping_cost  75000 non-null  float64
 6   order_status   75000 non-null  object 
dtypes: float64(1), object(6)
memory usage: 4.0+ MB


In [29]:
order_validation = orders_df.merge(
    customers_df[
        ["customer_id", "signup_date"]
    ],
    on="customer_id",
    how="left"
)

order_validation["order_date"] = pd.to_datetime(
    order_validation["order_date"]
)

order_validation["signup_date"] = pd.to_datetime(
    order_validation["signup_date"]
)

invalid_orders = order_validation[
    order_validation["order_date"]
    <
    order_validation["signup_date"]
]

print(
    f"Orders before customer signup: "
    f"{len(invalid_orders):,}"
)

Orders before customer signup: 0


In [30]:
orders_df["order_status"].value_counts(
    normalize=True
).round(3)

order_status
Completed    0.97
Cancelled    0.03
Name: proportion, dtype: float64

In [31]:
orders_df["device"].value_counts(
    normalize=True
).round(3)

device
Mobile     0.579
Desktop    0.361
Tablet     0.059
Name: proportion, dtype: float64

In [32]:
orders_check = orders_df.copy()

orders_check["order_date"] = pd.to_datetime(
    orders_check["order_date"]
)

orders_check.groupby(
    orders_check["order_date"].dt.year
)["order_id"].count()

order_date
2023    21160
2024    24945
2025    28895
Name: order_id, dtype: int64

## 4. Order Item Data

The order items table represents the individual products contained within each customer order.

One order may contain multiple products.

Each order item contains:
- Unique order item ID
- Order ID
- Product ID
- Quantity purchased
- Unit price
- Discount percentage

This table will later be combined with the orders and products tables to calculate revenue, cost of goods sold, gross profit, and product-level margins.

In [33]:
# Create realistic product popularity weights

category_popularity = {
    "Electronics": 1.30,
    "Fashion": 1.20,
    "Home & Kitchen": 1.00,
    "Beauty & Health": 0.95,
    "Sports & Outdoors": 0.85
}

products_df["popularity_weight"] = (
    products_df["category"].map(category_popularity)
    * np.random.uniform(
        0.60,
        1.40,
        size=len(products_df)
    )
)

product_probabilities = (
    products_df["popularity_weight"]
    / products_df["popularity_weight"].sum()
)

product_ids = products_df["product_id"].to_numpy()

In [34]:
order_items = []

product_price_lookup = (
    products_df
    .set_index("product_id")["list_price"]
)

item_id = 1

for order_id in orders_df["order_id"]:

    # Number of different products in an order
    number_of_items = np.random.choice(
        [1, 2, 3, 4],
        p=[0.55, 0.28, 0.12, 0.05]
    )

    # Select unique products using popularity weights
    selected_products = np.random.choice(
        product_ids,
        size=number_of_items,
        replace=False,
        p=product_probabilities
    )

    for product_id in selected_products:

        list_price = product_price_lookup[product_id]

        quantity = random.choices(
            [1, 2, 3, 4],
            weights=[72, 20, 6, 2]
        )[0]

        discount_pct = random.choices(
            [0.00, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40],
            weights=[40, 15, 15, 10, 10, 7, 3]
        )[0]

        order_items.append({
            "order_item_id": f"ITEM-{item_id:07d}",
            "order_id": order_id,
            "product_id": product_id,
            "quantity": quantity,
            "unit_price": list_price,
            "discount_pct": discount_pct
        })

        item_id += 1


order_items_df = pd.DataFrame(order_items)

print(
    f"Order items generated: "
    f"{len(order_items_df):,}"
)

Order items generated: 125,273


In [35]:
order_items_df.head(10)

,order_item_id,order_id,product_id,quantity,unit_price,discount_pct
0,ITEM-0000001,ORD-000001,PROD-0124,1,225.92,0.30
1,ITEM-0000002,ORD-000001,PROD-0045,1,233.26,0.05
2,ITEM-0000003,ORD-000001,PROD-0058,1,31.18,0.00
3,ITEM-0000004,ORD-000002,PROD-0012,1,302.76,0.30
4,ITEM-0000005,ORD-000002,PROD-0115,2,47.22,0.00
5,ITEM-0000006,ORD-000003,PROD-0139,1,251.16,0.00
6,ITEM-0000007,ORD-000004,PROD-0109,1,309.23,0.00
7,ITEM-0000008,ORD-000004,PROD-0032,1,129.41,0.10
8,ITEM-0000009,ORD-000005,PROD-0147,1,36.00,0.00
9,ITEM-0000010,ORD-000005,PROD-0092,3,146.11,0.30


In [36]:
order_items_df.shape

(125273, 6)

In [37]:
order_items_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125273 entries, 0 to 125272
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_item_id  125273 non-null  object 
 1   order_id       125273 non-null  object 
 2   product_id     125273 non-null  object 
 3   quantity       125273 non-null  int64  
 4   unit_price     125273 non-null  float64
 5   discount_pct   125273 non-null  float64
dtypes: float64(2), int64(1), object(3)
memory usage: 5.7+ MB


In [38]:
invalid_order_ids = (
    ~order_items_df["order_id"]
    .isin(orders_df["order_id"])
).sum()

print(
    f"Order items with invalid order IDs: "
    f"{invalid_order_ids:,}"
)

Order items with invalid order IDs: 0


In [39]:
invalid_product_ids = (
    ~order_items_df["product_id"]
    .isin(products_df["product_id"])
).sum()

print(
    f"Order items with invalid product IDs: "
    f"{invalid_product_ids:,}"
)

Order items with invalid product IDs: 0


In [40]:
order_items_check = order_items_df.merge(
    products_df[
        [
            "product_id",
            "category",
            "unit_cost",
            "list_price"
        ]
    ],
    on="product_id",
    how="left"
)

In [41]:
order_items_check["gross_sales"] = (
    order_items_check["quantity"]
    * order_items_check["unit_price"]
)

order_items_check["discount_amount"] = (
    order_items_check["gross_sales"]
    * order_items_check["discount_pct"]
)

order_items_check["net_sales"] = (
    order_items_check["gross_sales"]
    - order_items_check["discount_amount"]
)

order_items_check["cogs"] = (
    order_items_check["quantity"]
    * order_items_check["unit_cost"]
)

order_items_check["gross_profit"] = (
    order_items_check["net_sales"]
    - order_items_check["cogs"]
)

In [42]:
category_check = (
    order_items_check
    .groupby("category")
    .agg(
        units_sold=("quantity", "sum"),
        net_sales=("net_sales", "sum"),
        gross_profit=("gross_profit", "sum")
    )
    .round(2)
    .sort_values(
        "net_sales",
        ascending=False
    )
)

category_check

,units_sold,net_sales,gross_profit
category,,,
Electronics,40507,7053531.89,3128153.94
Home & Kitchen,40493,6364440.16,2729869.11
Beauty & Health,34218,6057884.48,2568519.83
Sports & Outdoors,25022,3967903.35,1645157.47
Fashion,33132,3864187.47,1569911.62


## 5. Returns Data

The returns table represents products returned by customers after purchase.

Each return contains:
- Unique return ID
- Order item ID
- Return date
- Return reason
- Refund amount

Return probability will vary by product category to create realistic differences in return behavior.

This will allow us to analyze:
- Return rate
- Refund impact
- Product and category return patterns
- Profitability after returns

In [43]:
category_return_rates = {
    "Electronics": 0.08,
    "Fashion": 0.15,
    "Home & Kitchen": 0.07,
    "Beauty & Health": 0.05,
    "Sports & Outdoors": 0.08
}

In [44]:
return_candidates = (
    order_items_df
    .merge(
        products_df[
            ["product_id", "category"]
        ],
        on="product_id",
        how="left"
    )
    .merge(
        orders_df[
            ["order_id", "order_date", "order_status"]
        ],
        on="order_id",
        how="left"
    )
)

return_candidates["order_date"] = pd.to_datetime(
    return_candidates["order_date"]
)

In [45]:
returns = []

return_reasons = [
    "Wrong Size",
    "Damaged",
    "Not as Expected",
    "Changed Mind",
    "Wrong Item",
    "Late Delivery"
]

return_id = 1

for _, row in return_candidates.iterrows():

    # Cancelled orders cannot be returned
    if row["order_status"] != "Completed":
        continue

    return_probability = category_return_rates[
        row["category"]
    ]

    if np.random.random() < return_probability:

        days_until_return = np.random.randint(
            2,
            31
        )

        return_date = (
            row["order_date"]
            + pd.Timedelta(days=days_until_return)
        )

        gross_sales = (
            row["quantity"]
            * row["unit_price"]
        )

        refund_amount = (
            gross_sales
            * (1 - row["discount_pct"])
        )

        returns.append({
            "return_id": f"RET-{return_id:06d}",
            "order_item_id": row["order_item_id"],
            "return_date": return_date.date(),
            "return_reason": random.choice(
                return_reasons
            ),
            "refund_amount": round(
                refund_amount,
                2
            )
        })

        return_id += 1


returns_df = pd.DataFrame(returns)

print(
    f"Returns generated: "
    f"{len(returns_df):,}"
)

Returns generated: 10,269


In [46]:
returns_df.head(10)

,return_id,order_item_id,return_date,return_reason,refund_amount
0,RET-000001,ITEM-0000005,2024-12-13,Wrong Item,94.44
1,RET-000002,ITEM-0000025,2025-09-19,Not as Expected,359.28
2,RET-000003,ITEM-0000028,2025-12-31,Changed Mind,110.00
3,RET-000004,ITEM-0000031,2024-03-25,Not as Expected,59.51
4,RET-000005,ITEM-0000047,2025-07-12,Late Delivery,37.78
5,RET-000006,ITEM-0000049,2025-07-18,Damaged,146.11
6,RET-000007,ITEM-0000050,2024-06-06,Wrong Item,265.34
7,RET-000008,ITEM-0000069,2024-09-24,Wrong Size,93.10
8,RET-000009,ITEM-0000085,2024-04-15,Damaged,116.89
9,RET-000010,ITEM-0000102,2025-08-11,Wrong Item,37.68


In [47]:
returns_df.shape

(10269, 5)

In [48]:
returns_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10269 entries, 0 to 10268
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   return_id      10269 non-null  object 
 1   order_item_id  10269 non-null  object 
 2   return_date    10269 non-null  object 
 3   return_reason  10269 non-null  object 
 4   refund_amount  10269 non-null  float64
dtypes: float64(1), object(4)
memory usage: 401.3+ KB


In [49]:
invalid_return_items = (
    ~returns_df["order_item_id"]
    .isin(order_items_df["order_item_id"])
).sum()

print(
    f"Returns with invalid order item IDs: "
    f"{invalid_return_items:,}"
)

Returns with invalid order item IDs: 0


In [50]:
return_check = (
    return_candidates[
        return_candidates["order_status"]
        == "Completed"
    ]
    .merge(
        returns_df[
            ["order_item_id", "return_id"]
        ],
        on="order_item_id",
        how="left"
    )
)

return_check["was_returned"] = (
    return_check["return_id"].notna()
)

In [51]:
category_returns = (
    return_check
    .groupby("category")
    .agg(
        items_sold=("order_item_id", "count"),
        items_returned=("was_returned", "sum"),
        return_rate=("was_returned", "mean")
    )
)

category_returns["return_rate"] = (
    category_returns["return_rate"]
    * 100
)

category_returns.round(2).sort_values(
    "return_rate",
    ascending=False
)

,items_sold,items_returned,return_rate
category,,,
Fashion,23225,3418,14.72
Electronics,28570,2283,7.99
Sports & Outdoors,17540,1394,7.95
Home & Kitchen,28301,1973,6.97
Beauty & Health,23889,1201,5.03


## 6. Marketing Data

The marketing table represents daily campaign performance across the company's acquisition channels.

Each record contains:
- Marketing ID
- Date
- Marketing channel
- Campaign name
- Marketing spend
- Impressions
- Clicks

This dataset will later allow us to calculate:
- Click-through rate
- Customer acquisition cost
- Revenue by acquisition channel
- Return on ad spend
- Marketing efficiency

In [52]:
marketing_channels = [
    "Paid Search",
    "Social Media",
    "Email",
    "Affiliate",
    "Display Ads",
    "Organic Search"
]

In [53]:
marketing = []

marketing_id = 1

marketing_dates = pd.date_range(
    START_DATE,
    END_DATE,
    freq="D"
)

for date in marketing_dates:

    for channel in marketing_channels:

        # Different channels have different spending profiles
        if channel == "Paid Search":
            spend = np.random.uniform(900, 2400)

        elif channel == "Social Media":
            spend = np.random.uniform(600, 1800)

        elif channel == "Display Ads":
            spend = np.random.uniform(300, 1100)

        elif channel == "Affiliate":
            spend = np.random.uniform(200, 800)

        elif channel == "Email":
            spend = np.random.uniform(50, 300)

        else:  # Organic Search
            spend = 0


        # Simulate impressions
        impressions = np.random.randint(
            10_000,
            180_000
        )


        # Different channels have different CTR behavior
        ctr_ranges = {
            "Paid Search": (0.025, 0.060),
            "Social Media": (0.015, 0.045),
            "Email": (0.030, 0.080),
            "Affiliate": (0.020, 0.050),
            "Display Ads": (0.005, 0.020),
            "Organic Search": (0.025, 0.065)
        }

        ctr = np.random.uniform(
            *ctr_ranges[channel]
        )

        clicks = int(
            impressions * ctr
        )


        marketing.append({
            "marketing_id": f"MKT-{marketing_id:07d}",
            "date": date.date(),
            "channel": channel,
            "campaign": f"{channel} Campaign",
            "spend": round(spend, 2),
            "impressions": impressions,
            "clicks": clicks
        })

        marketing_id += 1


marketing_df = pd.DataFrame(marketing)

print(
    f"Marketing records generated: "
    f"{len(marketing_df):,}"
)

Marketing records generated: 6,576


In [54]:
marketing_df.head(12)

,marketing_id,date,channel,campaign,spend,impressions,clicks
0,MKT-0000001,2023-01-01,Paid Search,Paid Search Campaign,1766.02,43436,1942
1,MKT-0000002,2023-01-01,Social Media,Social Media Campaign,1758.19,20766,618
2,MKT-0000003,2023-01-01,Email,Email Campaign,134.61,136117,9855
3,MKT-0000004,2023-01-01,Affiliate,Affiliate Campaign,426.71,129218,3757
4,MKT-0000005,2023-01-01,Display Ads,Display Ads Campaign,472.64,15356,141
5,MKT-0000006,2023-01-01,Organic Search,Organic Search Campaign,0.00,39946,1149
6,MKT-0000007,2023-01-02,Paid Search,Paid Search Campaign,1050.87,116755,4675
7,MKT-0000008,2023-01-02,Social Media,Social Media Campaign,1161.59,111874,2836
8,MKT-0000009,2023-01-02,Email,Email Campaign,248.78,18423,1239
9,MKT-0000010,2023-01-02,Affiliate,Affiliate Campaign,366.57,41838,1313


In [55]:
marketing_df.shape

(6576, 7)

In [56]:
marketing_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6576 entries, 0 to 6575
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   marketing_id  6576 non-null   object 
 1   date          6576 non-null   object 
 2   channel       6576 non-null   object 
 3   campaign      6576 non-null   object 
 4   spend         6576 non-null   float64
 5   impressions   6576 non-null   int64  
 6   clicks        6576 non-null   int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 359.8+ KB


In [57]:
marketing_summary = (
    marketing_df
    .groupby("channel")
    .agg(
        total_spend=("spend", "sum"),
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum")
    )
)

In [58]:
marketing_summary["ctr_pct"] = (
    marketing_summary["clicks"]
    / marketing_summary["impressions"]
    * 100
)

marketing_summary.round(2).sort_values(
    "total_spend",
    ascending=False
)

,total_spend,impressions,clicks,ctr_pct
channel,,,,
Paid Search,1793952.23,104497142,4415502,4.23
Social Media,1331907.28,102292284,3074029,3.01
Display Ads,766022.80,106047388,1311250,1.24
Affiliate,547603.89,104658603,3630963,3.47
Email,187637.20,105691366,5835719,5.52
Organic Search,0.00,102001564,4523713,4.43


## 7. Create Raw Source Datasets

The underlying generated datasets are internally consistent.

To simulate a realistic business environment, selected data-quality issues will now be introduced into copies of the original tables.

These raw datasets will represent the files received from the company and will later be cleaned and validated in a separate analysis notebook.

In [59]:
raw_customers = customers_df.copy()
raw_products = products_df.copy()
raw_orders = orders_df.copy()
raw_order_items = order_items_df.copy()
raw_returns = returns_df.copy()
raw_marketing = marketing_df.copy()

In [60]:
# Missing cities
missing_city_idx = raw_customers.sample(
    frac=0.01,
    random_state=10
).index

raw_customers.loc[
    missing_city_idx,
    "city"
] = np.nan


# Missing age groups
missing_age_idx = raw_customers.sample(
    frac=0.005,
    random_state=11
).index

raw_customers.loc[
    missing_age_idx,
    "age_group"
] = np.nan


# Add duplicate customer records
customer_duplicates = raw_customers.sample(
    n=25,
    random_state=12
)

raw_customers = pd.concat(
    [raw_customers, customer_duplicates],
    ignore_index=True
)

In [61]:
# Inconsistent category naming

product_error_idx = (
    raw_products[
        raw_products["category"] == "Home & Kitchen"
    ]
    .sample(
        frac=0.20,
        random_state=13
    )
    .index
)

raw_products.loc[
    product_error_idx,
    "category"
] = "Home and Kitchen"

In [62]:
missing_device_idx = raw_orders.sample(
    frac=0.005,
    random_state=14
).index

raw_orders.loc[
    missing_device_idx,
    "device"
] = np.nan

In [63]:
shipping_outlier_idx = raw_orders.sample(
    n=20,
    random_state=15
).index

raw_orders.loc[
    shipping_outlier_idx,
    "shipping_cost"
] = 99.99

In [64]:
missing_reason_idx = raw_returns.sample(
    frac=0.03,
    random_state=16
).index

raw_returns.loc[
    missing_reason_idx,
    "return_reason"
] = np.nan

In [65]:
social_idx = (
    raw_marketing[
        raw_marketing["channel"] == "Social Media"
    ]
    .sample(
        frac=0.10,
        random_state=17
    )
    .index
)

raw_marketing.loc[
    social_idx,
    "channel"
] = "social media"

In [66]:
missing_click_idx = raw_marketing.sample(
    frac=0.003,
    random_state=18
).index

raw_marketing.loc[
    missing_click_idx,
    "clicks"
] = np.nan

In [67]:
raw_datasets = {
    "customers": raw_customers,
    "products": raw_products,
    "orders": raw_orders,
    "order_items": raw_order_items,
    "returns": raw_returns,
    "marketing": raw_marketing
}

In [68]:
for name, df in raw_datasets.items():
    print(
        f"{name:<12} "
        f"{df.shape[0]:>8,} rows | "
        f"{df.shape[1]} columns"
    )

customers      25,025 rows | 7 columns
products          150 rows | 7 columns
orders         75,000 rows | 7 columns
order_items   125,273 rows | 6 columns
returns        10,269 rows | 5 columns
marketing       6,576 rows | 7 columns


In [70]:
for name, df in raw_datasets.items():

    file_path = RAW_DIR / f"{name}.csv"

    df.to_csv(
        file_path,
        index=False
    )

    print(
        f"{name}.csv saved successfully"
    )

customers.csv saved successfully
products.csv saved successfully
orders.csv saved successfully
order_items.csv saved successfully
returns.csv saved successfully
marketing.csv saved successfully


In [71]:
list(RAW_DIR.iterdir())

[PosixPath('/Users/gonzalojenkinsmadrid/E-Commerce-Analytics/data/raw/customers.csv'),
 PosixPath('/Users/gonzalojenkinsmadrid/E-Commerce-Analytics/data/raw/products.csv'),
 PosixPath('/Users/gonzalojenkinsmadrid/E-Commerce-Analytics/data/raw/orders.csv'),
 PosixPath('/Users/gonzalojenkinsmadrid/E-Commerce-Analytics/data/raw/marketing.csv'),
 PosixPath('/Users/gonzalojenkinsmadrid/E-Commerce-Analytics/data/raw/order_items.csv'),
 PosixPath('/Users/gonzalojenkinsmadrid/E-Commerce-Analytics/data/raw/returns.csv')]

## 8. Create SQLite Raw Database

The six raw CSV datasets will also be loaded into a SQLite database as staging tables.

The database will preserve the raw source data and will later be queried using SQL.

In [72]:
database_path = (
    DATABASE_DIR / "ecommerce.db"
)

connection = sqlite3.connect(
    database_path
)

In [73]:
for name, df in raw_datasets.items():

    table_name = f"raw_{name}"

    df.to_sql(
        table_name,
        connection,
        if_exists="replace",
        index=False
    )

    print(
        f"{table_name} loaded successfully"
    )

raw_customers loaded successfully
raw_products loaded successfully
raw_orders loaded successfully
raw_order_items loaded successfully
raw_returns loaded successfully
raw_marketing loaded successfully


In [74]:
connection.close()

print(
    f"SQLite database created at:\n"
    f"{database_path}"
)

SQLite database created at:
/Users/gonzalojenkinsmadrid/E-Commerce-Analytics/database/ecommerce.db


In [75]:
database_path.exists()

True

In [76]:
connection = sqlite3.connect(
    database_path
)

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    connection
)

connection.close()

tables

,name
0,raw_customers
1,raw_marketing
2,raw_order_items
3,raw_orders
4,raw_products
5,raw_returns
